## Python 1: Assignment 1 Part 2

In [8]:
import pandas as pd
import numpy as np

### Exercise 1

In [9]:
ba = pd.read_csv('StockData/BA.csv')
# print(ba.info())
# # NB y not Y since originsl fsts id 2-digit year not 4-digit year
ba['Date'] = pd.to_datetime(ba['Date'], format='%m/%d/%y') # or format=r''
ba.set_index(ba['Date'], inplace=True)
print(ba.info())

# read data in again, setting index as we do so
# ba = pd.read_csv('StockData/BA.csv', header=0, index_col='Date', parse_dates=True, date_format='%m/%d/%y')

# ba.info()

ba.head(10)
ba.tail(3)

<class 'pandas.DataFrame'>
DatetimeIndex: 1259 entries, 2013-10-01 to 2018-09-28
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       1259 non-null   datetime64[us]
 1   Open       1259 non-null   float64       
 2   High       1259 non-null   float64       
 3   Low        1259 non-null   float64       
 4   Close      1259 non-null   float64       
 5   Adj Close  1259 non-null   float64       
 6   Volume     1259 non-null   int64         
dtypes: datetime64[us](1), float64(5), int64(1)
memory usage: 78.7 KB
None


,Date,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,,
2018-09-26,2018-09-26,367.500000,368.559998,364.320007,365.029999,365.029999,2368600
2018-09-27,2018-09-27,368.399994,368.970001,364.399994,367.390015,367.390015,2355900
2018-09-28,2018-09-28,368.630005,374.220001,366.220001,371.899994,371.899994,3595800


### Exercise 2

In [50]:
sp500 = pd.read_csv('StockData/SP500.csv', header=0, index_col=0, parse_dates=True, date_format='%m/%d/%y')
ba = pd.read_csv('StockData/BA.csv', header=0, index_col=0, parse_dates=True, date_format='%m/%d/%y')

sp500oc = sp500[['Open','Close']].copy()
ba_oc = ba[['Open','Close']].copy()

sp500oc['Close_1'] = sp500oc['Close'].shift(1)
ba_oc['Close_1'] = ba_oc['Close'].shift(1)

sp500oc.head()
# ba_oc.head()

sp500oc['rtns'] = (sp500oc['Close'] - sp500oc['Close_1'])/sp500oc['Close_1']
sp500oc['log_rtns'] = np.log(sp500oc['Close']) - np.log(sp500oc['Close_1'])

ba_oc['rtns'] = (ba_oc['Close'] - ba_oc['Close_1'])/ba_oc['Close_1']
ba_oc['log_rtns'] = np.log(ba_oc['Close']) - np.log(ba_oc['Close_1'])

sp500oc.head()
ba_oc.head()

merged_df = sp500oc.merge(ba_oc, how='left', left_index=True, right_index=True, suffixes =('_sp','_ba'))
merged_df.head()

print('Mean SP500 Daily Return: {:%}'.format(np.mean(merged_df['rtns_sp'])))
print('Mean SP500 Daily Log-Return: {:%}'.format(np.mean(merged_df['log_rtns_sp'])))
print('Mean BA Daily Return: {:%}'.format(np.mean(merged_df['rtns_ba'])))
print('Mean BA Daily Log-Return: {:%}'.format(np.mean(merged_df['log_rtns_ba'])))

merged_df.describe()

Mean SP500 Daily Return: 0.046694%
Mean SP500 Daily Log-Return: 0.043670%
Mean BA Daily Return: 0.101280%
Mean BA Daily Log-Return: 0.091420%


,Open_sp,Close_sp,Close_1_sp,rtns_sp,log_rtns_sp,Open_ba,Close_ba,Close_1_ba,rtns_ba,log_rtns_ba
count,1260.000000,1260.000000,1259.000000,1259.000000,1259.000000,1259.000000,1259.000000,1258.000000,1258.000000,1258.000000
mean,2208.119022,2208.579183,2208.018896,0.000467,0.000437,181.936466,182.014718,181.863776,0.001013,0.000914
std,311.783587,311.699505,311.187978,0.007758,0.007771,77.847511,77.860511,77.706980,0.014015,0.014020
min,1656.989990,1655.449951,1655.449951,-0.040979,-0.041843,105.120003,108.440002,108.440002,-0.089290,-0.093531
25%,1984.504975,1985.515015,1985.489990,-0.002826,-0.002830,130.544998,130.590004,130.580005,-0.006296,-0.006316
50%,2101.550049,2101.265014,2101.040039,0.000489,0.000489,143.220001,143.029999,143.019997,0.001135,0.001134
75%,2434.364991,2434.777527,2434.145019,0.004560,0.004550,199.599998,199.799995,199.574997,0.009019,0.008979
max,2936.760010,2930.750000,2930.750000,0.039034,0.038291,372.000000,372.230011,372.230011,0.098795,0.094214


### Exercise 3

In [60]:
import pandas as pd

sp500 = pd.read_csv('StockData/SP500.csv', header=0, index_col=0, parse_dates=True, date_format='%m/%d/%y')
ba = pd.read_csv('StockData/BA.csv', header=0, index_col=0, parse_dates=True, date_format='%m/%d/%y')

sp500oc = sp500[['Open','Close']].copy()
ba_oc = ba[['Open','Close']].copy()

sp500oc['Close_1'] = sp500oc['Close'].shift(1)
ba_oc['Close_1'] = ba_oc['Close'].shift(1)

sp500oc.head()
ba_oc.head()

sp500oc['Pos_Day'] = sp500oc['Close'] > sp500oc['Open']
ba_oc['Pos_Day'] = ba_oc['Close'] > ba_oc['Open']

merged_df = sp500oc.merge(ba_oc, how='left', left_index=True, right_index=True, suffixes =('_sp','_ba'))
merged_df.head()

merged_df['ba_pos_sp_neg'] = merged_df.apply(
    lambda x: 1 if (x['Pos_Day_ba'] == True) and (x['Pos_Day_sp'] == False) else 0, axis=1)
merged_df.head(10)

merged_df.dropna(inplace=True)
merged_df['ba_pos_sp_neg'].value_counts()

ba_pos_sp_neg
0    1072
1     186
Name: count, dtype: int64

### Exercise 4

In [71]:
# Joining with a gap in the index
bbus = pd.read_csv('StockData/BB/BB_NYSE.csv', index_col='Date', parse_dates=True, date_format='%m/%d/%y')
bbus.head()
bbcad = pd.read_csv('StockData/BB/BB_TO.csv', index_col='Date', parse_dates=True, date_format='%m/%d/%y')
bbcad.head()

# Subset to only Jan
bbus = bbus.loc['2018-01']
bbus
bbcad = bbcad.loc['2018-01']
bbcad
bb = bbcad.join(bbus, lsuffix='_tse', rsuffix='_nyse')
bb.loc['2018-01-12':'2018-01-17']

,Open_tse,High_tse,Low_tse,Close_tse,Adj Close_tse,Volume_tse,Open_nyse,High_nyse,Low_nyse,Close_nyse,Adj Close_nyse,Volume_nyse
Date,,,,,,,,,,,,
2018-01-12,16.830000,17.16,16.820000,17.00,17.00,2020200,13.47,13.73,13.45,13.65,13.65,6310900.0
2018-01-15,17.750000,17.75,17.250000,17.48,17.48,2057500,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-16,18.000000,18.07,16.870001,17.18,17.18,6180100,14.37,14.55,13.55,13.81,13.81,16808400.0
2018-01-17,17.299999,17.35,16.930000,17.00,17.00,3020400,13.91,13.98,13.57,13.67,13.67,6013100.0


### Exercise 5

In [77]:
sp500 = pd.read_csv('StockData/SP500.csv', header=0, index_col=0, parse_dates=True, date_format='%m/%d/%y')
sp500.columns = [i.lower().replace(' ','_') for i in sp500.columns]
sp500.head()

sp500['adj_close_1'] = sp500['adj_close'].shift()
sp500['log_rtn_c_to_o'] = np.log(sp500['open']) - np.log(sp500['adj_close_1'])
sp500['log_rtn_o_to_c'] = np.log(sp500['adj_close']) - np.log(sp500['open'])

# When finding daily vol, the numbers can become really small decimals.
# We can multiply the returns by 100 before squaring if a numerical problem occurs
sp500['vol_c_to_o'] = (sp500['log_rtn_c_to_o'] ** 2)
sp500['vol_o_to_c'] = (sp500['log_rtn_o_to_c'] ** 2)

vol_weight = 0.5

sp500['vol'] = sp500['vol_o_to_c'] * vol_weight + sp500['vol_c_to_o'] * (1-vol_weight)

sp500['vol_1'] = sp500['vol'].shift(1)
sp500['vol_ma_5'] = sp500['vol_1'].rolling(window = 5, min_periods=5).mean()
sp500['vol_ma_21'] = sp500['vol_1'].rolling(window = 21, min_periods=21).mean()

# save the dataFrame to a file
# NB make sure 'output' folder already exists
sp500.to_csv("Output/SP500ex5.csv")
sp500.to_excel("Output/SP500ex5.xlsx")


sp500.head(30)

,open,high,low,close,adj_close,volume,adj_close_1,log_rtn_c_to_o,log_rtn_o_to_c,vol_c_to_o,vol_o_to_c,vol,vol_1,vol_ma_5,vol_ma_21
Date,,,,,,,,,,,,,,,
2013-09-30,1687.260010,1687.260010,1674.989990,1681.550049,1681.550049,3308630000,NaN,NaN,-0.003390,NaN,1.149143e-05,NaN,NaN,NaN,NaN
2013-10-01,1682.410034,1696.550049,1682.069946,1695.000000,1695.000000,3238690000,1681.550049,0.000511,0.007455,2.614208e-07,5.558346e-05,2.792244e-05,NaN,NaN,NaN
2013-10-02,1691.900024,1693.869995,1680.339966,1693.869995,1693.869995,3148600000,1695.000000,-0.001831,0.001164,3.350982e-06,1.354144e-06,2.352563e-06,2.792244e-05,NaN,NaN
2013-10-03,1692.349976,1692.349976,1670.359985,1678.660034,1678.660034,3279650000,1693.869995,-0.000898,-0.008122,8.059865e-07,6.597021e-05,3.338810e-05,2.352563e-06,NaN,NaN
2013-10-04,1678.790039,1691.939941,1677.329956,1690.500000,1690.500000,2880270000,1678.660034,0.000077,0.006951,5.997372e-09,4.831674e-05,2.416137e-05,3.338810e-05,NaN,NaN
2013-10-07,1687.150024,1687.150024,1674.699951,1676.119995,1676.119995,2678490000,1690.500000,-0.001984,-0.006559,3.934725e-06,4.302223e-05,2.347848e-05,2.416137e-05,NaN,NaN
2013-10-08,1676.219971,1676.790039,1655.030029,1655.449951,1655.449951,3569230000,1676.119995,0.000060,-0.012468,3.557586e-09,1.554609e-04,7.773222e-05,2.347848e-05,0.000022,NaN
2013-10-09,1656.989990,1662.469971,1646.469971,1656.400024,1656.400024,3577840000,1655.449951,0.000930,-0.000356,8.646244e-07,1.268145e-07,4.957194e-07,7.773222e-05,0.000032,NaN
2013-10-10,1660.880005,1692.560059,1660.880005,1692.560059,1692.560059,3362300000,1656.400024,0.002701,0.018895,7.295391e-06,3.570069e-04,1.821511e-04,4.957194e-07,0.000032,NaN


### Exercise 6

In [103]:
ba = pd.read_csv('StockData/BA.csv', header=0, index_col=0, parse_dates=True, date_format='%m/%d/%y')
ba.columns = [i.lower().replace(' ','_') for i in ba.columns]
ba.head()

financials = pd.read_csv('StockData/fundamentals.csv', index_col=0)
financials.columns = [i.lower().replace(' ','_') for i in financials.columns]
# financials.info()
# financials

# # Convert to month end to handle alignment with reporting date
# # using the previous quarter end to not have information early
ba['match_date'] = ba.index + pd.offsets.MonthEnd(-3)

financials['period_ending'] = pd.to_datetime(financials['period_ending'], format='%m/%d/%y')
financials['match_date'] = financials['period_ending'] + pd.offsets.MonthEnd(0)

financials['ROA'] = financials['net_income']/financials['total_assets']

max_date = max(financials[financials['ticker_symbol']=='BA']['match_date'])
min_date = min(financials[financials['ticker_symbol']=='BA']['match_date'])

ba_fin = ba.merge(financials[financials['ticker_symbol']=='BA'][['match_date','ROA']], how='left')
ba_fin.info()
ba_fin
ba_fin['ROA'] = ba_fin['ROA'].ffill()
ba_fin['ROA']

ba_fin = ba_fin[ba_fin['match_date'] <= (max_date + pd.DateOffset(years=1))]
ba_fin = ba_fin[ba_fin['match_date'] >= (min_date)]
ba_fin

<class 'pandas.DataFrame'>
RangeIndex: 1259 entries, 0 to 1258
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   open        1259 non-null   float64       
 1   high        1259 non-null   float64       
 2   low         1259 non-null   float64       
 3   close       1259 non-null   float64       
 4   adj_close   1259 non-null   float64       
 5   volume      1259 non-null   int64         
 6   match_date  1259 non-null   datetime64[us]
 7   ROA         88 non-null     float64       
dtypes: datetime64[us](1), float64(6), int64(1)
memory usage: 78.8 KB


,open,high,low,close,adj_close,volume,match_date,ROA
104,127.180000,128.440002,126.519997,128.220001,113.064377,4847100,2013-12-31,0.049480
105,129.380005,131.000000,129.330002,130.229996,114.836815,5422800,2013-12-31,0.049480
106,130.399994,130.800003,128.690002,128.789993,113.567009,4095200,2013-12-31,0.049480
107,129.009995,129.369995,128.029999,128.860001,113.628731,3595900,2013-12-31,0.049480
108,129.750000,130.160004,128.070007,128.539993,113.346573,4297800,2013-12-31,0.049480
...,...,...,...,...,...,...,...,...
1127,320.149994,331.140015,318.320007,321.000000,317.836151,7237500,2017-12-31,0.054391
1128,330.480011,331.940002,323.589996,328.970001,325.727600,5392900,2017-12-31,0.054391
1129,331.510010,334.880005,319.000000,321.119995,317.954987,5355500,2017-12-31,0.054391
1130,321.850006,323.239990,311.170013,320.019989,316.865814,5634700,2017-12-31,0.054391


### Exercise 7

In [123]:
#Open FF3 - Fama French Three Factors and CAT
ba = pd.read_csv('StockData/ba.csv', header=0, index_col=0, parse_dates = True, date_format='%m/%d/%y')
ba.columns = [i.lower().replace(' ','_') for i in ba.columns]
ba.head()

# NB no dat parsing here
ff3 = pd.read_csv('StockData/FF3_monthly.csv', header=0, index_col=0)
# the dates in the FF3 file cannot be automatically parsed
# will be imported as a int64index

# # The index is not date time
ff3.info()

# # Create a temporary date column and copy the index
ff3['date'] = ff3.index

# # Parse the date time, default is beginning of the month, offset to the end of the month
ff3['date'] = pd.to_datetime(ff3.index,format='%Y%m') + pd.offsets.MonthEnd(0)
ff3
ff3.set_index(['date'], drop=True, inplace=True)
ff3.head()

# # Same rules as previously defined
ohlc_rule = {'open':'first', 'high':'max', 'low':'min',
            'close':'last', 'volume':'sum', 'adj_close':'last'}

# # Resample the daily data into monthly data using resample and agg
ba_mon = ba.resample('ME').agg(ohlc_rule)
# ba_mon.to_csv('output/ba_mon.csv')
ba_mon.head()

# # Log Returns on adjust close
ba_mon['log_rtns'] = np.log(ba_mon['adj_close']) - np.log(ba_mon['adj_close'].shift(1))

# # Merge on the index, no need for suffix because there are no duplicate columns
ba_ff = pd.merge(ba_mon, ff3, how='left', left_index=True, right_index=True)
ba_ff.head()
# # Notice how the returns are multiplied by 100 for the FF3 data, correct this for ba
ba_ff['log_rtns'] = ba_ff['log_rtns'] * 100
ba_ff.head()
# ba_ff.info()
# ba_ff.to_csv('output/BA_FF3_mon.csv')

<class 'pandas.DataFrame'>
Index: 1107 entries, 192607 to 201809
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Mkt-RF  1107 non-null   float64
 1   SMB     1107 non-null   float64
 2   HML     1107 non-null   float64
 3   RF      1107 non-null   float64
dtypes: float64(4)
memory usage: 43.2 KB


,open,high,low,close,volume,adj_close,log_rtns,Mkt-RF,SMB,HML,RF
Date,,,,,,,,,,,
2013-10-31,117.970001,131.880005,113.339996,130.500000,102075200,114.013985,NaN,4.18,-1.49,1.14,0.0
2013-11-30,130.899994,142.000000,130.580002,134.250000,88250600,117.717674,3.196805,3.12,1.24,0.24,0.0
2013-12-31,134.990005,138.880005,130.550003,136.490005,78680000,119.681831,1.654765,2.81,-0.50,-0.31,0.0
2014-01-31,136.009995,144.570007,123.080002,125.260002,113971400,109.834763,-8.585973,-3.32,0.86,-2.09,0.0
2014-02-28,124.239998,130.940002,118.769997,128.919998,119986800,113.681641,3.442484,4.65,0.32,-0.40,0.0
